# Document Splitting (v2 — fixed article-boundary detection)

In [ ]:
pip install langchain langchain-text-splitters langchain-core transformers

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

## Splitter mechanics

In [2]:
chunk_size = 26
chunk_overlap = 4

r_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
c_splitter = CharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)

In [3]:
text1 = 'abcdefghijklmnopqrstuvwxyz'
r_splitter.split_text(text1)

['abcdefghijklmnopqrstuvwxyz']

In [4]:
text2 = 'abcdefghijklmnopqrstuvwxyzabcdefg'
r_splitter.split_text(text2)

['abcdefghijklmnopqrstuvwxyz', 'wxyzabcdefg']

### Arabic example

In [5]:
arabic_text = "المادة الأولى يعد باطلا كل شرط يخالف احكام هذا القانون ما لم يكن الشرط اكثر فائدة للعامل"
r_splitter.split_text(arabic_text)

['المادة الأولى يعد باطلا كل',
 'كل شرط يخالف احكام هذا',
 'هذا القانون ما لم يكن',
 'يكن الشرط اكثر فائدة',
 'للعامل']

## Real project data — one chunk per article / per case

In [ ]:
import json
import re
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

PROCESSED_DIR = Path("../data/processed")

# sjc.bh case type codes (see 01_scraping/01a)
SJC_TYPE_CODES = {"M": "مدني", "J": "جنائي", "S": "شرعي", "T": "تجاري", "E": "انتخابات", "P": "توحيد المبادئ"}

# Both with and without the "ال" definite-article prefix — the corpus uses both.
ORDINAL_WORDS_DEF = ["الاولي", "الأولى", "الثانية", "الثالثة", "الرابعة", "الخامسة",
                     "السادسة", "السابعة", "الثامنة", "التاسعة", "العاشرة"]
ORDINAL_WORDS_BARE = ["اولي", "أولى", "ثانية", "ثالثة", "رابعة", "خامسة",
                      "سادسة", "سابعة", "ثامنة", "تاسعة", "عاشرة"]
ORDINAL_WORDS = ORDINAL_WORDS_DEF + ORDINAL_WORDS_BARE
ORDINAL_TO_DIGIT = {}
for _i, (_def, _bare) in enumerate(zip(ORDINAL_WORDS_DEF, ORDINAL_WORDS_BARE)):
    ORDINAL_TO_DIGIT[_def] = str(_i + 1)
    ORDINAL_TO_DIGIT[_bare] = str(_i + 1)
ORDINAL_TO_DIGIT["الأولى"] = "1"
ORDINAL_TO_DIGIT["أولى"] = "1"

# Corpus mixes ASCII hyphen and en-dash article-number wrapping; treat as interchangeable.
DASH = r"[-–—]"

# Compound numbers captured fully; accepts parens, dash-wrapped, bare digit, or ordinal word.
NUMBER_PART = (
    r"(?:\(\s*(\d+(?:\.\d+)*)\s*\)"
    r"|" + DASH + r"\s*(\d+(?:\.\d+)*)\s*" + DASH +
    r"|(\d+(?:\.\d+)*)"
    r"|" + "|".join(ORDINAL_WORDS) + r")"
)
ARTICLE_HEADER = re.compile(r"(?:المادة|مادة)\s*" + NUMBER_PART)

# A real article header starts a fresh sentence or heading, never mid-sentence.
SENTENCE_END = re.compile(r"[.:!؟\n]" + DASH + r"?\s*\Z")  # optional trailing dash covers "قرر الآتي:-"

# Chapter/section heading detector; requires a word boundary before الفصل to avoid matching
# "والفصل" (dismissal).
HEADING_MARKER = re.compile(r"(?:\A|\s)(?:الباب|الفصل|القسم)\s+\S+\s*[^.:!؟]{0,40}\Z")

# Segments under this size get merged into their neighbor.
MIN_SEGMENT_CHARS = 40


def article_no_from_match(m: re.Match) -> str | None:
    for g in m.groups():
        if g and (g.isdigit() or "." in g):
            return g
    matched_text = m.group(0)
    for word, digit in sorted(ORDINAL_TO_DIGIT.items(), key=lambda kv: -len(kv[0])):
        if word in matched_text:
            return digit
    return None


def is_real_header(text: str, m: re.Match) -> bool:
    """True if match is a real article heading, not a citation buried in prose."""
    before = text[:m.start()]
    if before.strip() == "":
        return True
    if SENTENCE_END.search(before):
        return True
    return bool(HEADING_MARKER.search(before))


def merge_tiny_segments(segments: list[dict], min_chars: int) -> list[dict]:
    """Merge any segment under min_chars into the segment that follows it."""
    if not segments:
        return segments
    out = [dict(segments[0])]
    for seg in segments[1:]:
        if len(out[-1]["text"]) < min_chars:
            out[-1]["text"] = (out[-1]["text"] + " " + seg["text"]).strip()
        else:
            out.append(dict(seg))
    if len(out) > 1 and len(out[-1]["text"]) < min_chars:
        last = out.pop()
        out[-1]["text"] = (out[-1]["text"] + " " + last["text"]).strip()
    return out


def segment_legislation_by_article(text: str) -> list[dict]:
    """One segment per real article header; citation-only matches excluded, tiny leftovers merged."""
    all_matches = list(ARTICLE_HEADER.finditer(text))
    matches = [m for m in all_matches if is_real_header(text, m)]
    if not matches:
        return [{"text": text, "article_no": None}]
    segments = []
    for i, m in enumerate(matches):
        start = m.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        segments.append({"text": text[start:end].strip(), "article_no": article_no_from_match(m)})
    return merge_tiny_segments(segments, MIN_SEGMENT_CHARS)


# BGE-M3 context limit is 8192 tokens; 6000 leaves a safety margin. Used only for lloc's fallback split.
MAX_TOKENS = 6000
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")
fallback_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer, chunk_size=MAX_TOKENS, chunk_overlap=200, separators=["\n\n", ". ", " ", ""]
)


def n_tokens(text: str) -> int:
    return len(tokenizer.encode(text, add_special_tokens=False))


def load_lloc_documents(path: Path) -> list[Document]:
    records = json.loads(path.read_text(encoding="utf-8"))
    docs = []
    for r in records:
        if not r.get("normalized_text"):
            continue
        base_meta = {
            "source": "lloc",
            "doc_id": r["code"],
            "title": r.get("title") or "",
            "categories": ", ".join(r.get("categories", [])),
        }
        for seg in segment_legislation_by_article(r["normalized_text"]):
            docs.append(Document(page_content=seg["text"], metadata={**base_meta, "article_no": seg["article_no"]}))
    return docs


SJC_NO_JUDGMENT_PLACEHOLDER = "مجموعة الاحكام الصادرة من محكمة التمييز لا يوجد"


def load_judgment_documents(path: Path, source: str, id_field: str) -> list[Document]:
    """Skips sjc.bh's placeholder text for cases with no judgment, and deduplicates exact-duplicate
    case-number records."""
    records = json.loads(path.read_text(encoding="utf-8"))
    docs = []
    skipped_placeholder = 0
    skipped_duplicate_id = 0
    seen_ids = set()
    for r in records:
        if not r.get("normalized_text"):
            continue
        if r["normalized_text"].strip() == SJC_NO_JUDGMENT_PLACEHOLDER:
            skipped_placeholder += 1
            continue
        doc_id = r.get(id_field) or r.get("key") or r.get("case_id")
        if doc_id in seen_ids:
            skipped_duplicate_id += 1
            continue
        seen_ids.add(doc_id)
        metadata = {"source": source, "doc_id": doc_id}
        if source == "sjc" and doc_id:
            parts = doc_id.split(" ")
            type_code = parts[1] if len(parts) > 1 else None
            metadata["case_type"] = SJC_TYPE_CODES.get(type_code)
        docs.append(Document(page_content=r["normalized_text"], metadata=metadata))
    if skipped_placeholder:
        print(f"{source}: skipped {skipped_placeholder} 'no judgment available' placeholder records")
    if skipped_duplicate_id:
        print(f"{source}: skipped {skipped_duplicate_id} exact-duplicate case-number records")
    return docs


lloc_docs = load_lloc_documents(PROCESSED_DIR / "lloc_normalized.json")
sjc_docs = load_judgment_documents(PROCESSED_DIR / "sjc_normalized.json", "sjc", "key")
ccb_docs = load_judgment_documents(PROCESSED_DIR / "ccb_normalized.json", "ccb", "case_id")

print(f"lloc: {len(lloc_docs)} article-segments")
print(f"sjc:  {len(sjc_docs)} per-case documents")
print(f"ccb:  {len(ccb_docs)} per-case documents")

missing_type = sum(1 for d in sjc_docs if not d.metadata.get("case_type"))
print(f"sjc documents missing case_type: {missing_type}")


### Legislation — one chunk per article, fallback split only if oversized

In [ ]:
def split_with_fallback(doc: Document) -> list[Document]:
    """One chunk per document by default; subdivided only if it exceeds MAX_TOKENS. Sub-chunks
    keep the parent's full metadata plus a sub_chunk_index."""
    if n_tokens(doc.page_content) <= MAX_TOKENS:
        return [doc]
    pieces = fallback_splitter.split_text(doc.page_content)
    return [
        Document(page_content=piece, metadata={**doc.metadata, "sub_chunk_index": i})
        for i, piece in enumerate(pieces)
    ]


lloc_splits = []
lloc_oversized = 0
for d in lloc_docs:
    pieces = split_with_fallback(d)
    if len(pieces) > 1:
        lloc_oversized += 1
    lloc_splits.extend(pieces)

with_article_no = sum(1 for d in lloc_splits if d.metadata.get("article_no"))
print(f"{len(lloc_docs)} article-segments -> {len(lloc_splits)} chunks "
      f"({lloc_oversized} articles needed a fallback split, {with_article_no} chunks tagged with an article_no)")
lloc_splits[1].page_content[:300]

### Judgments/rulings — always one chunk per whole case, no fragmentation

In [8]:
sjc_splits = sjc_docs
ccb_splits = ccb_docs

print(f"sjc: {len(sjc_docs)} cases -> {len(sjc_splits)} chunks (always 1:1, no fragmentation)")
print(f"ccb: {len(ccb_docs)} cases -> {len(ccb_splits)} chunks (always 1:1, no fragmentation)")


sjc: 9001 cases -> 9001 chunks (always 1:1, no fragmentation)
ccb: 93 cases -> 93 chunks (always 1:1, no fragmentation)


### Save the split documents for the next stage

In [ ]:
import pickle

all_splits = lloc_splits + sjc_splits + ccb_splits
OUT_PKL = Path("../data/processed/document_splits_v2.pkl")
OUT_PKL.write_bytes(pickle.dumps(all_splits))
print(f"Saved {len(all_splits)} total chunks -> {OUT_PKL}")

# Written to a new file so the current live vectorstore is left untouched until swapped in.
OUT_JSON = Path("../data/processed/document_splits_v2.json")
records = [{"page_content": d.page_content, "metadata": d.metadata} for d in all_splits]
OUT_JSON.write_text(json.dumps(records, ensure_ascii=False), encoding="utf-8")
print(f"Saved {len(records)} total chunks -> {OUT_JSON}")